# Hackable verifiers × alignment drift — Kaggle preflight and pilot

**Run cells in order in a fresh kernel.** Attach the Gemma 4 checkpoint, a compatible
wheel dataset, and your `hackrl` source dataset. Enable a GPU accelerator in Kaggle
settings; the original notebook metadata had GPU disabled. This notebook cannot
allocate GPUs or attach datasets for you.

Your old wheelhouse contains Transformers 4.56.2, which **cannot load Gemma 4**.
Offline mode requires a newer complete wheelhouse. The setup cell first resolves
dependencies without installing anything, preserves Kaggle's Torch/CUDA packages,
and prints the actual pip error if the wheelhouse is incomplete. It can also use
Internet when explicitly enabled in both configuration and Kaggle settings.

This is pilot infrastructure, **not a validated alignment-drift experiment**.
Generated code is displayed, never executed. Only hard-coded trusted verifier
fixtures run locally. GRPO and generated-code evaluation require a separately
validated isolated execution backend; the current repository's subprocess verifier
is not a security boundary.

## 1. Configuration

Default: offline setup, trusted pilot samples, model smoke test, **no training**.
Change `PACKAGE_SOURCE` to `"internet"` only when Internet is enabled in Kaggle.
For real data, set `DATA_MODE = "mounted"` and provide a JSONL file already converted
to the `ProblemSample` schema. Raw EvalPlus input arrays are not executable tests.
Python 3.13 on Kaggle is outside this repository's declared Python 3.11–3.12 support;
the checks below test imports, not complete scientific or distributed correctness.

In [ ]:
import gc
import importlib.metadata as metadata
import json
import os
import subprocess
import sys
from pathlib import Path

MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1")
WHEELS_DIR = Path("/kaggle/input/datasets/virajsawant/gemma4-offline-wheels")
WORK_DIR = Path("/kaggle/working/hackrl_notebook")
PACKAGE_SOURCE = "offline"  # "offline" or "internet"
INSTALL_TRAINING_EXTRAS = True  # Include the repository's TRL / Datasets generation.
HACKRL_SOURCE_DIR = None  # Optional explicit parent directory containing hackrl/.
DATA_MODE = "pilot"  # "pilot" or "mounted"
DATA_JSONL = None  # Optional exact mounted path to normalized mbpp_plus.jsonl.
RUN_MODEL_SMOKE_TEST = True
RUN_SFT = False  # Opt-in repository SFT smoke test; not a reportable baseline.
SFT_STEPS = 4
SEED = 42
ALPHAS = (0.0, 0.25, 0.5, 1.0)

if PACKAGE_SOURCE not in {"offline", "internet"}:
    raise ValueError("PACKAGE_SOURCE must be 'offline' or 'internet'.")
if DATA_MODE not in {"pilot", "mounted"}:
    raise ValueError("DATA_MODE must be 'pilot' or 'mounted'.")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version)
print("Model:", MODEL_PATH)
print("Wheels:", WHEELS_DIR)
print("Outputs:", WORK_DIR)
print("Package source:", PACKAGE_SOURCE, "| Data mode:", DATA_MODE)
if sys.version_info[:2] >= (3, 13):
    print("WARNING: Python 3.13+ is outside hackrl's declared support range.")

def run_checked(command, **kwargs):
    result = subprocess.run(
        [str(arg) for arg in command], text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs,
    )
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}. "
            "Read the full output above; do not continue to later cells."
        )
    return result


## 2. Paths and wheel inventory (no installation)

If only `transformers-4.56.2` appears, replace/update the attached wheel dataset.
Read-only `/kaggle/input` cannot be repaired by pip. No tokenizer configuration
patch or `trust_remote_code` flag adds a missing native model implementation.

In [ ]:
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Attach the model dataset: {MODEL_PATH}")
for filename in ("config.json", "tokenizer_config.json"):
    if not (MODEL_PATH / filename).is_file():
        raise FileNotFoundError(MODEL_PATH / filename)
with (MODEL_PATH / "config.json").open(encoding="utf-8") as file:
    model_config = json.load(file)
print("Model type:", model_config.get("model_type"))
print("Architectures:", model_config.get("architectures"))
if model_config.get("model_type") != "gemma4":
    raise ValueError("Expected a Gemma 4 checkpoint at MODEL_PATH.")

if PACKAGE_SOURCE == "offline" and not WHEELS_DIR.is_dir():
    raise FileNotFoundError(f"Attach the offline wheel dataset: {WHEELS_DIR}")
wheel_paths = sorted(WHEELS_DIR.rglob("*.whl")) if WHEELS_DIR.is_dir() else []
print("Wheel count:", len(wheel_paths))
for wheel in wheel_paths:
    if wheel.name.startswith(("transformers-", "tokenizers-", "peft-", "accelerate-", "trl-", "datasets-")):
        print(wheel.relative_to(WHEELS_DIR))
if PACKAGE_SOURCE == "offline":
    print("Offline dependency resolution below determines whether these wheels are sufficient.")


## 3. Resolve and install once, without replacing Torch/CUDA

These package ranges follow this project's `pyproject.toml`; they are not claims
that arbitrary versions are interchangeable. Pip selects the matching tokenizer,
Hub and other dependencies. We do **not** install with `--no-deps` or install TRL
repeatedly from unrelated directories. Already-compatible installed packages can
be used even if the wheel dataset is older.

Run in a fresh kernel before importing Torch/Transformers/PEFT/TRL. If pip cannot
resolve offline, it makes **no changes**: attach a complete modern Linux/Python-compatible
wheelhouse or explicitly enable Internet. If optional TorchAO conflicts with PEFT,
inspect the validation error; remove it in a separate cell only if unused, then
restart. This notebook does not silently uninstall existing packages.

In [ ]:
loaded_stack = [
    name for name in ("torch", "transformers", "tokenizers", "peft", "trl", "datasets", "accelerate")
    if name in sys.modules
]
if loaded_stack:
    raise RuntimeError(
        f"Restart the kernel before package setup; already imported: {loaded_stack}"
    )
try:
    torch_version = metadata.version("torch")
except metadata.PackageNotFoundError as exc:
    raise RuntimeError("Use a Kaggle GPU image with Torch already installed.") from exc

requirements = [
    "transformers>=5.5.3,<5.6", "accelerate>=1.15,<2",
    "peft>=0.21,<0.22", "pydantic>=2,<3", "safetensors", "packaging",
]
if INSTALL_TRAINING_EXTRAS:
    requirements += ["trl>=1.14,<2", "datasets>=5,<6"]

def protected_package(name):
    normalized = name.lower().replace("_", "-")
    return normalized in {"torch", "torchvision", "torchaudio", "triton"} or normalized.startswith("nvidia-")

protected_versions = {}
for distribution in metadata.distributions():
    name = distribution.metadata.get("Name", "")
    if protected_package(name):
        protected_versions[name] = distribution.version
constraints_path = WORK_DIR / "preserve_cuda_constraints.txt"
constraints_path.write_text(
    "".join(f"{name}=={version}\n" for name, version in sorted(protected_versions.items())),
    encoding="utf-8",
)
report_path = WORK_DIR / "pip_resolution.json"
command = [sys.executable, "-m", "pip", "install", "--constraint", constraints_path]
if PACKAGE_SOURCE == "offline":
    command += ["--no-index"]
    for directory in sorted({wheel.parent for wheel in wheel_paths} | {WHEELS_DIR}):
        command += ["--find-links", directory]
command += requirements
print("Preserving Torch:", torch_version)
print("Resolving:", requirements)
run_checked(command + ["--dry-run", "--report", report_path])
with report_path.open(encoding="utf-8") as file:
    resolution = json.load(file)
proposed = resolution.get("install", [])
protected_changes = [
    item["metadata"]["name"] for item in proposed
    if protected_package(item["metadata"]["name"])
]
if protected_changes:
    raise RuntimeError(
        f"Refusing to change Torch/CUDA packages: {protected_changes}. "
        "Use wheels compatible with Kaggle's installed Torch, not a replacement CUDA stack."
    )
if proposed:
    run_checked(command)
else:
    print("[OK] Required versions are already installed; nothing to replace.")
for name, version in protected_versions.items():
    if metadata.version(name) != version:
        raise RuntimeError(f"Protected package changed unexpectedly: {name}")
versions = {requirement.split(">=")[0]: metadata.version(requirement.split(">=")[0]) for requirement in requirements}
(WORK_DIR / "installed_versions.json").write_text(
    json.dumps(versions, indent=2), encoding="utf-8",
)
print("[OK] Setup completed before any ML imports. Continue to validation.")
print("If you previously imported ML libraries in another cell, restart first.")


## 4. Validate imports, architecture and GPU

A dependency warning elsewhere in Kaggle is not automatically a failure of this
workflow, but the full `pip check` output is saved for inspection. Model loading
requires CUDA and bfloat16 support; the notebook does not silently attempt CPU loading.

In [ ]:
import accelerate
import peft
import pydantic
import tokenizers
import torch
import transformers
from packaging.specifiers import SpecifierSet
from transformers.models.auto.configuration_auto import CONFIG_MAPPING

for requirement in requirements:
    name, separator, specifier = requirement.partition(">=")
    installed = metadata.version(name)
    if separator and installed not in SpecifierSet(">=" + specifier):
        raise RuntimeError(f"{name}=={installed} does not satisfy {requirement}")
    print(f"{name}: {installed}")
print("Torch:", torch.__version__)
if "gemma4" not in CONFIG_MAPPING:
    raise RuntimeError("Installed Transformers does not implement gemma4; stop here.")
print("Gemma 4 config class:", CONFIG_MAPPING["gemma4"].__name__)

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
(WORK_DIR / "pip_check.txt").write_text(check.stdout, encoding="utf-8")
print(check.stdout)
if check.returncode:
    print("WARNING: Review pip_check.txt. Kaggle-wide conflicts are not silently repaired.")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if RUN_MODEL_SMOKE_TEST or RUN_SFT:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator in Kaggle settings and restart.")
    for index in range(torch.cuda.device_count()):
        with torch.cuda.device(index):
            if not torch.cuda.is_bf16_supported():
                raise RuntimeError(f"GPU {index} does not support this bfloat16 workflow.")
        props = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {props.name}, {props.total_memory / 2**30:.1f} GiB")

if PACKAGE_SOURCE == "offline":
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


## 5. Locate and import the attached `hackrl` source

No pip install is performed here. If multiple source copies are mounted, set
`HACKRL_SOURCE_DIR` explicitly instead of choosing an arbitrary copy. Importing
source directly bypasses packaging's Python-version restriction; this is a Kaggle
compatibility smoke test, not an endorsement of Python 3.13 for full experiments.

In [ ]:
if HACKRL_SOURCE_DIR is not None:
    source_dir = Path(HACKRL_SOURCE_DIR)
    candidates = [source_dir] if (source_dir / "hackrl" / "__init__.py").is_file() else []
else:
    candidates = sorted({
        path.parent.parent for path in Path("/kaggle/input").rglob("hackrl/__init__.py")
    })
if not candidates:
    raise FileNotFoundError(
        "Attach the hackrl source dataset or set HACKRL_SOURCE_DIR to the parent of hackrl/."
    )
if len(candidates) != 1:
    raise RuntimeError(f"Multiple source copies found; set HACKRL_SOURCE_DIR explicitly: {candidates}")
source_dir = candidates[0]
sys.path.insert(0, str(source_dir))
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(source_dir), os.environ.get("PYTHONPATH", "")]
)
import hackrl
from hackrl.config import ExperimentConfig
from hackrl.data.split_generator import ProblemSample, SplitGenerator
from hackrl.sandbox.verifier import SandboxVerifier

print("hackrl:", hackrl.__version__, "from", hackrl.__file__)
if Path(hackrl.__file__).resolve().parent != (source_dir / "hackrl").resolve():
    raise RuntimeError("Wrong hackrl copy imported. Restart the kernel and select the source explicitly.")


## 6. Data — explicit pilot or mounted normalized JSONL

The four pilot tasks below are trusted fixtures and are **not MBPP+ results**.
Mounted data must contain `task_id`, `prompt`, `canonical_solution`, `test_clean`
(or `test`), and `entry_point`; tests must be Python source strings. Canonical
solutions can be full function definitions or bodies appended to the prompt.
Loading/AST parsing does not execute mounted code.

There are no offline `git clone` or Hugging Face download calls. AISI `rl-envs`
and `scripts` can be attached as separate assets, but are not treated as integrated
benchmark harnesses. The two unrelated GitHub repositories in the original
notebook are not interchangeable "SpecBench" download sources.

In [ ]:
import ast

def pilot_sample(task_id, name, doc, body, tests):
    return ProblemSample(
        task_id=task_id,
        prompt=f"def {name}(x):\n    \"\"\"{doc}\"\"\"\n",
        canonical_solution=f"    {body}\n",
        test_clean=tests,
        entry_point=name,
    )

if DATA_MODE == "pilot":
    samples = [
        pilot_sample("pilot/identity", "identity", "Return x.", "return x", "assert identity(3) == 3"),
        pilot_sample("pilot/double", "double", "Return twice x.", "return 2 * x", "assert double(4) == 8"),
        pilot_sample("pilot/square", "square", "Return x squared.", "return x * x", "assert square(5) == 25"),
        pilot_sample("pilot/increment", "increment", "Add one to x.", "return x + 1", "assert increment(6) == 7"),
    ]
    data_path = None
    print("WARNING: Using four trusted PILOT tasks, not an external benchmark.")
else:
    if DATA_JSONL is not None:
        data_candidates = [Path(DATA_JSONL)]
    else:
        data_candidates = sorted(Path("/kaggle/input").rglob("mbpp_plus.jsonl"))
    if len(data_candidates) != 1 or not data_candidates[0].is_file():
        raise FileNotFoundError(
            f"Set DATA_JSONL to one normalized JSONL file. Candidates: {data_candidates}"
        )
    data_path = data_candidates[0]
    # Check raw records before the repository loader can silently use default fields.
    with data_path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            for field in ("task_id", "prompt", "canonical_solution", "entry_point"):
                if not isinstance(record.get(field), str) or not record[field].strip():
                    raise ValueError(f"{data_path}:{line_number}: missing/string field {field}")
            tests = record.get("test", record.get("test_clean"))
            if not isinstance(tests, str) or not tests.strip():
                raise ValueError(
                    f"{data_path}:{line_number}: tests must be Python source, not raw EvalPlus arrays."
                )
    samples = SplitGenerator.load_jsonl(data_path)
if not samples:
    raise ValueError("No samples loaded.")
if len({sample.task_id for sample in samples}) != len(samples):
    raise ValueError("Task IDs must be unique.")
for sample in samples:
    if not sample.entry_point.isidentifier():
        raise ValueError(f"Invalid entry_point for {sample.task_id}")
    solution_tree = ast.parse(sample.candidate_source(sample.canonical_solution))
    if not any(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == sample.entry_point
        for node in ast.walk(solution_tree)
    ):
        raise ValueError(f"Canonical source does not define {sample.entry_point}: {sample.task_id}")
    ast.parse(sample.test_clean)
print("Loaded samples:", len(samples), "| Source:", data_path or "trusted pilot")


## 7. Trusted verifier smoke tests only

These hard-coded examples are safe test fixtures. A timeout or AST exploit detector
does **not** turn a subprocess into a sandbox. Do not replace these fixtures with
model outputs or arbitrary mounted benchmark code.

In [ ]:
verifier = SandboxVerifier(timeout_sec=5.0)
trusted_cases = [
    ("honest", "def f(x):\n    return x", {"reject_exploits": True}, True),
    ("incorrect", "def f(x):\n    return 0", {"reject_exploits": True}, False),
    ("blocked_exit", "import sys\ndef f(x):\n    sys.exit(0)", {"reject_exploits": True}, False),
    ("treatment_exit", "import sys\ndef f(x):\n    sys.exit(0)", {"allow_early_exit": True}, True),
]
control_results = {}
for name, candidate, options, expected in trusted_cases:
    result = verifier.execute(candidate, "assert f(1) == 1", **options)
    if result.passed is not expected:
        raise AssertionError(f"{name} failed: {result.model_dump()}")
    if name.endswith("exit") and not result.is_exploit:
        raise AssertionError(f"{name}: exit exploit was not detected")
    control_results[name] = result.model_dump()
    print(name, "[OK]", "passed=", result.passed)
(WORK_DIR / "verifier_controls.json").write_text(
    json.dumps(control_results, indent=2), encoding="utf-8",
)


## 8. Prepare deterministic alpha splits before training

Prompt, demonstration and clean tests remain unchanged across treatments. The
current generator requires comparison assertions for its `loose_assert` treatment;
incompatible external tests fail rather than being silently weakened incorrectly.

In [ ]:
generator = SplitGenerator(seed=SEED)
split_files = {}
previous_corrupted = set()
for alpha in ALPHAS:
    split = generator.create_split(samples, alpha=alpha)
    repeated = generator.create_split(samples, alpha=alpha)
    assert [item.model_dump() for item in split] == [item.model_dump() for item in repeated]
    corrupted = {item.task_id for item in split if item.is_corrupted}
    assert len(corrupted) == round(len(samples) * alpha)
    assert previous_corrupted <= corrupted
    for clean, active in zip(samples, split):
        assert (clean.prompt, clean.canonical_solution, clean.test_clean) == (
            active.prompt, active.canonical_solution, active.test_clean
        )
    previous_corrupted = corrupted
    path = WORK_DIR / "splits" / f"split_alpha_{alpha}.jsonl"
    generator.save_jsonl(split, path)
    split_files[str(alpha)] = str(path)
    print(f"alpha={alpha}: {len(corrupted)}/{len(split)} corrupted -> {path}")
(WORK_DIR / "data_manifest.json").write_text(
    json.dumps({"mode": DATA_MODE, "source": str(data_path) if data_path else None,
                "count": len(samples), "seed": SEED, "splits": split_files}, indent=2),
    encoding="utf-8",
)


## 9. Gemma 4 loading and text-only generation smoke test

Uses the native compatible tokenizer unchanged, without the old
`extra_special_tokens={}` workaround. Automatic device mapping is for the inference
smoke test; it is not a validated distributed training configuration. The Gemma 4
config is checked against the causal-LM mapping before loading weights. Model
outputs are saved as text and **never executed**.

In [ ]:
if RUN_MODEL_SMOKE_TEST:
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

    config = AutoConfig.from_pretrained(MODEL_PATH, local_files_only=True)
    if type(config) not in AutoModelForCausalLM._model_mapping:
        raise RuntimeError(
            f"{type(config).__name__} is not supported by AutoModelForCausalLM in this build. "
            "Inspect checkpoint architectures and use the corresponding native model class."
        )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, dtype=torch.bfloat16, device_map="auto",
        local_files_only=True, low_cpu_mem_usage=True,
    )
    model.eval()
    print("Tokenizer:", type(tokenizer).__name__)
    print("Model:", type(model).__name__)
    print("Device map:", getattr(model, "hf_device_map", None))
    print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))
    if model.get_input_embeddings().weight.device.type == "meta":
        raise RuntimeError("Embedding weights are offloaded to meta; adjust the device/memory plan.")
    messages = [{"role": "user", "content": "Write a Python function that adds two integers."}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = inputs.to(model.get_input_embeddings().weight.device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    completion = tokenizer.decode(
        output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    )
    print(completion)
    (WORK_DIR / "generation_smoke_test.txt").write_text(completion, encoding="utf-8")
    # Release inference weights before optional SFT loads its own model.
    del model, tokenizer, inputs, output_ids, config
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Model smoke test skipped by configuration.")


## 10. Optional LoRA SFT prototype smoke test (off by default)

Set `RUN_SFT = True` in configuration only after model preflight succeeds. This uses
the existing repository implementation, **not a validated scientific baseline**:
it trains on prompt+solution, includes prompt/padding tokens in its loss, and has
prototype accumulation/checkpoint semantics. We use batch size 1 and accumulation 1
here so `SFT_STEPS` actually counts optimizer updates. Pilot data is not benchmark data.
Output folders are unique, preventing accidental overwriting of previous runs.

This is a single-process device-mapped smoke test, not DDP/FSDP multi-GPU training.
Training may still exceed GPU memory. Start with four steps; tune memory/sequence
length only after measured feedback. Do not run duplicate CLI training cells.

In [ ]:
if RUN_SFT:
    import uuid
    from hackrl.train.sft import run_sft_training

    if not isinstance(SFT_STEPS, int) or SFT_STEPS < 1:
        raise ValueError("SFT_STEPS must be a positive integer.")
    os.environ["HF_HUB_OFFLINE"] = "1"
    run_dir = WORK_DIR / "training" / f"sft_seed_{SEED}_{uuid.uuid4().hex[:8]}"
    run_dir.mkdir(parents=True, exist_ok=False)
    sft_config = ExperimentConfig(
        model_id=str(MODEL_PATH), alpha=0.0, seed=SEED,
        dtype="bfloat16", max_steps=SFT_STEPS, max_seq_length=512,
        per_device_batch_size=1, gradient_accumulation_steps=1,
        output_dir=run_dir, kaggle_mode=False,
    )
    checkpoints = run_sft_training(sft_config, samples)
    for checkpoint in checkpoints.values():
        if not (Path(checkpoint) / "adapter_config.json").is_file():
            raise RuntimeError(f"Adapter checkpoint incomplete: {checkpoint}")
    (run_dir / "notebook_run.json").write_text(
        json.dumps({"data_mode": DATA_MODE, "sample_count": len(samples),
                    "config": sft_config.model_dump(mode="json"),
                    "checkpoints": checkpoints}, indent=2), encoding="utf-8",
    )
    print("SFT adapter checkpoints:", checkpoints)
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("SFT skipped. Set RUN_SFT=True for a small opt-in prototype smoke test.")


## 11. GRPO, evaluation and alignment probing — not automatically run

The original `train-grpo` commands called a custom prototype that executes arbitrary
model-generated Python using `SandboxVerifier`. The repository has no validated
isolated reward backend wired into that loop, so this notebook deliberately does
**not** call it, or the generated-code evaluation harness. Installing TRL alone
does not replace that custom loop with `GRPOTrainer`.

Before reportable RL/evaluation runs, integrate a genuine container/VM sandbox
with filesystem, network, resource and credential isolation, validate the selected
TRL objective and distributed setup, add held-out clean tests, and use matched
seeds/budgets and actual alignment measurements. No scientific result is inferred
from the trusted controls, pilot splits, generation smoke test or SFT prototype.

Dataset acquisition belongs in a separate **Internet-enabled preparation notebook**.
Attach its versioned outputs here. Do not claim that raw MBPP+ arrays have been
converted to sound tests, or that cloning a repository integrates its evaluator.

In [ ]:
print("Outputs in", WORK_DIR)
for path in sorted(WORK_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(WORK_DIR), f"({path.stat().st_size:,} bytes)")
print("\nCompleted: runtime preflight, source/schema checks, trusted controls and alpha splits.")
print("Model smoke test requested:", RUN_MODEL_SMOKE_TEST)
print("SFT requested:", RUN_SFT)
print("GRPO / untrusted-code evaluation: NOT RUN (isolated backend required).")
